In [7]:
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field
from typing import List

class DatasetRow(BaseModel):
    program_name: str = Field(description="Official name of the academic program")
    program_level: str = Field(description="UG, PG, Diploma, Certificate, or PhD level")
    domain: str = Field(description="Broad academic domain such as Engineering, Science, Management, Humanities, etc.")
    eligibility: str = Field(description="Eligibility criteria required for admission")
    description: str = Field(description="Short academic description of the program")
    skills_learned: List[str] = Field(
        description="Key academic or professional skills gained after completing the program"
    )
    career_outcomes: List[str] = Field(
        description="Typical career paths or job roles after completing the program"
    )

model = ChatOllama(model="llama3.2", temperature=0).with_structured_output(DatasetRow)

system_prompt="""
You are an academic data normalization and enrichment engine.
Your task is to transform raw university program information into a clean,
structured, and minimal semantic representation suitable for machine learning
and recommendation systems.
Follow ALL rules strictly.
--------------------------------
GENERAL RULES
--------------------------------
1. Use ONLY the information provided in the input.
2. Do NOT hallucinate facts, rankings, or specializations.
3. Keep wording academically neutral and precise.
4. Output MUST be valid JSON matching the required schema.
5. Do NOT include explanations, notes, or extra text outside JSON.
--------------------------------
FIELD-LEVEL TRANSFORMATION RULES
-------------------------------
program_name:
- Preserve the official academic program name exactly.
- Do not shorten or paraphrase.
program_level:
- Must be a SINGLE WORD chosen from:
  ["Certificate", "Diploma", "UG", "PG", "PhD"]
- Infer from the program title if necessary.
domain:
- Must be a SINGLE WORD academic domain such as:
  Engineering, Science, Management, Humanities, Commerce, Law, Education, Pharmacy, ComputerScience, etc.
- Choose the most appropriate broad discipline.
eligibility:
- Rewrite into a **simple, clear, single-sentence requirement**.
- Remove unnecessary wording, percentages, and legal phrasing.
- Keep only the **minimum academic qualification and subject requirement**.
- Example:
  "Bachelor’s degree in Biology or related field."
description:
- Write a **2–3 sentence academic summary** of what the program studies.
- Focus on knowledge areas, learning scope, and discipline.
- Do NOT mention university names, rankings, or admissions process.
skills_learned:
- Provide **3 to 6 concise skill phrases**.
- Each item must be SHORT (2–4 words).
- Example:
  ["Data analysis", "Laboratory techniques", "Research methodology"]
career_outcomes:
- Provide **3 to 6 clear and descriptive career paths**.
- Each item should be a **readable professional role**.
- Example:
  ["Research scientist in biotechnology", "Clinical laboratory specialist"]
"""

In [8]:
import pandas as pd
import time
from langchain_core.messages import HumanMessage, SystemMessage
dataset = pd.read_excel("courses_data.xlsx")
results = []

for idx, row in dataset.iterrows():
    try:
        info = f"""
    Program Name: {row.get('Program Name', '')}
    Program Level: {row.get('Program Level', '')}
    Faculty/Domain: {row.get('Name Of Faculty', '')}
    Eligibility: {row.get('Eligibility', '')}
    """
        conversation = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=info)
        ]
        response: DatasetRow = model.invoke(conversation)

        results.append(response.model_dump())
        print(f"Processed row: {idx+1}")
        time.sleep(0.5)
    except Exception as e:
        print(f"Error at row: {idx+1}")

final_dataset = pd.DataFrame(results).to_csv("final_data.csv")

Processed row: 1
Processed row: 2
Processed row: 3
Processed row: 4
Processed row: 5
Processed row: 6
Processed row: 7
Processed row: 8
Processed row: 9
Processed row: 10
Processed row: 11
Processed row: 12
Processed row: 13
Processed row: 14
Processed row: 15
Processed row: 16
Processed row: 17
Processed row: 18
Processed row: 19
Processed row: 20
Processed row: 21
Processed row: 22
Processed row: 23
Processed row: 24
Processed row: 25
Processed row: 26
Processed row: 27
Processed row: 28
Processed row: 29
Processed row: 30
Processed row: 31
Processed row: 32
Processed row: 33
Processed row: 34
Processed row: 35
Processed row: 36
Processed row: 37
Processed row: 38
Processed row: 39
Processed row: 40
Processed row: 41
Processed row: 42
Processed row: 43
Processed row: 44
Processed row: 45
Processed row: 46
Processed row: 47
Processed row: 48
Processed row: 49
Processed row: 50
Processed row: 51
Processed row: 52
Processed row: 53
Processed row: 54
Processed row: 55
Processed row: 56
P

In [6]:
import pandas as pd
df = pd.read_csv("final_data.csv")
df.to_excel("file.xlsx")